<a href="https://colab.research.google.com/github/amasha-sinhalagoda/J26-IT-334/blob/Landslide-PreProcess-Dataset/Landslide_DataPreprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install unrar tool and extract the .rar file
!apt-get install unrar -y
!unrar x "NuwaraEliya(SLIIT)-(Bagya).rar"

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:7.0.7-1build1).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.

UNRAR 7.00 freeware      Copyright (c) 1993-2024 Alexander Roshal


Extracting from NuwaraEliya(SLIIT)-(Bagya).rar

Extracting  DATA_SHARING_GUIDE.pdf                                         0%  OK 
Creating    Dithawa_HR_By_District                                    OK
Creating    Dithawa_HR_By_District/Nuwara_Eliya                       OK
Extracting  Dithawa_HR_By_District/Nuwara_Eliya/Nuwara_Eliya_Dithawa_HR_Inspections.cpg       0%  OK 
Extracting  Dithawa_HR_By_District/Nuwara_Eliya/Nuwara_Eliya_Dithawa_HR_Inspections.dbf       0%  OK 
Extracting  Dithawa_HR_By_District/Nuwara_Eliya/Nuwara_Eliya_Dithawa_HR_Inspections.prj       0%  OK 
Extracting  Dithawa_HR_By_District/Nuwara_Eliya/Nuwara_Eliya_Dithawa_HR_Inspections.shp      

In [ ]:
# =========================================================
# STEP 1: INSTALL REQUIRED EXCEL & GIS LIBRARIES
# =========================================================
!pip install geopandas pandas openpyxl xlsxwriter shapely -q

import geopandas as gpd
import pandas as pd
import numpy as np
import os
from google.colab import files

print("Dependencies loaded successfully!")

# =========================================================
# STEP 2: LOCATE & LOAD EXTRACTED SHAPEFILES
# =========================================================
inspections_shp = None
lhmp_shp = None

# Automatically scan directories to locate extracted .shp files
for root, dirs, files_list in os.walk("."):
    for file in files_list:
        if "Dithawa" in file and file.endswith(".shp"):
            inspections_shp = os.path.join(root, file)
        elif "LHMP" in file and file.endswith(".shp"):
            lhmp_shp = os.path.join(root, file)

print(f"Inspections shapefile found: {inspections_shp}")
print(f"LHMP shapefile found: {lhmp_shp}")

# Load shapefiles into GeoDataFrames and convert to Sri Lanka Grid 1999 (EPSG:5235)
gdf_inspections = gpd.read_file(inspections_shp).to_crs(epsg=5235)
gdf_lhmp = gpd.read_file(lhmp_shp).to_crs(epsg=5235)

print(f"Loaded {len(gdf_inspections)} inspection points.")
print(f"Loaded {len(gdf_lhmp)} LHMP hazard polygons.")

# =========================================================
# STEP 3: FEATURE EXTRACTION & DATA PREPROCESSING
# =========================================================
# Extract spatial metric coordinates (X, Y)
gdf_inspections["X_Coord"] = gdf_inspections.geometry.x
gdf_inspections["Y_Coord"] = gdf_inspections.geometry.y

# Assign binary positive target label (1 = Landslide / High Risk Point)
gdf_inspections["Landslide_Occurrence"] = 1

# Map risk ratings (HR1, HR2, HR3) to numerical severity ranks
def parse_hr_level(val):
    if pd.isna(val): return 0
    val_str = str(val).upper()
    if "HR1" in val_str: return 3
    if "HR2" in val_str: return 2
    if "HR3" in val_str: return 1
    return 0

# Identify risk column and encode
risk_cols = [c for c in gdf_inspections.columns if "RISK" in c.upper() or "HR" in c.upper()]
if risk_cols:
    gdf_inspections["Severity_Rank"] = gdf_inspections[risk_cols[0]].apply(parse_hr_level)
else:
    gdf_inspections["Severity_Rank"] = 1

# Perform Spatial Join: overlay LHMP hazard polygon attributes onto point locations
joined_gdf = gpd.sjoin(gdf_inspections, gdf_lhmp, how="left", predicate="intersects")

# Extract Hazard Zone category field
hazard_cols = [c for c in joined_gdf.columns if "HAZARD" in c.upper() or "ZONE" in c.upper()]
if hazard_cols:
    gdf_inspections["NBRO_Hazard_Zone"] = joined_gdf[hazard_cols[0]].fillna("Unmapped")
else:
    gdf_inspections["NBRO_Hazard_Zone"] = "Unmapped"

# Create clean tabular DataFrame by removing vector geometry
df_ml_matrix = pd.DataFrame(gdf_inspections.drop(columns=["geometry"]))

# =========================================================
# STEP 4: WRITE & DOWNLOAD MULTI-SHEET EXCEL (.XLSX)
# =========================================================
excel_filename = "Landslide_Dataset_Preprocessed.xlsx"

with pd.ExcelWriter(excel_filename, engine="xlsxwriter") as writer:
    # Sheet 1: Preprocessed ML Features
    df_ml_matrix.to_excel(writer, sheet_name="ML_Feature_Matrix", index=False)

    # Sheet 2: Unprocessed Raw Attributes
    pd.DataFrame(gdf_inspections.drop(columns=["geometry"])).to_excel(
        writer, sheet_name="Raw_Attributes", index=False
    )

    # Sheet 3: Dataset Summary Information
    summary_df = pd.DataFrame({
        "Attribute": ["Record Count", "CRS Projection", "Positive Target Ratio", "Total Columns"],
        "Value": [len(df_ml_matrix), "EPSG:5235 (Sri Lanka Grid)", "1.0 (Positive Instances)", len(df_ml_matrix.columns)]
    })
    summary_df.to_excel(writer, sheet_name="Metadata_Summary", index=False)

print(f"\nSUCCESS: Excel dataset '{excel_filename}' generated.")

# Trigger browser file download prompt
files.download(excel_filename)